In [1]:
import os
import time
import torch
import skimage
import sklearn.metrics
import torchvision
import wandb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
# from torchvision.ops import sigmoid_focal_loss

import mnds
import detection
import vision_transformer as vit

In [2]:
SCALE_FACTOR = 1.0
PATCH_SIZE = 256
STRIDE = 8
FEATURE_SIZE = 384
TOKENS_PER_PATCH = PATCH_SIZE // STRIDE
STEP = 16
EPOCHS = 20
THRESHOLD = 0.5

LOSS_FN = 'combined'
LR = 1e-5
BATCH_SIZE = 32
FINETUNE = True
WEIGHT_DECAY = 1e-6


CURRENT_PATH = os.getcwd()
DIRECTORY = CURRENT_PATH + '/dataset_v2'
OUTPUT_DIR = "/model_output/output/"

In [3]:
files = os.listdir(DIRECTORY)
filelist = [file for file in files if not file.startswith('.')] # avoid files starting with . when untarring in CHTC
annot_files = [x for x in filelist if x.endswith('png')]
annot_files.sort()
# annot_files = annot_files[0:10] # using all 18 images


i = 0
training_files = annot_files.copy()
validation_files = [annot_files[i]]
del training_files[i]
print(len(training_files))

17


In [4]:
training_set = mnds.MicronucleiDataset(
    filelist=training_files, 
    directory=DIRECTORY, 
    mode="random",
    edges=True,
    transform=mnds.detection_transforms,
    scale_factor=SCALE_FACTOR,
    patch_size=PATCH_SIZE
)

100%|██████████| 17/17 [01:03<00:00,  3.71s/it]


In [5]:
crop, mask = next(iter(training_set))

# Check if interpolation messed up the binary mask values

In [8]:
np.unique(mask)

array([0., 1.], dtype=float32)

In [11]:
np.unique(crop)

array([0.0000000e+00, 3.7252903e-08, 7.2829425e-07, ..., 9.9971837e-01,
       9.9985379e-01, 1.0000000e+00], dtype=float32)